# Phase 3 — Hybrid Search + Evaluation

Goal: make retrieval better, then measure how much better. Phase 2 left `hybrid_search` (pgvector + Postgres full-text, fused with RRF) as the retrieval baseline — this phase adds a cross-encoder reranker on top, then (next) RAGAS + LLM-as-judge evaluation comparing baseline vs hybrid vs hybrid+rerank.

This notebook starts with the reranker — the first open item in[docs/phase-2-rag-pipeline.md](../docs/phase-2-rag-pipeline.md)'s Phase 3 backlog.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

## Cross-encoder reranker

`hybrid_search` returns its top-k by RRF-fused rank — a cheap combination of two independent scores (lexical + vector), not a model that actually reads the query and the chunk together. A cross-encoder does: it scores each (query, chunk) pair jointly, which is slower per pair but more accurate at judging relevance. Standard pattern: retrieve a wider top-k cheaply, rerank down to a narrower top-k with the cross-encoder.

`src/rerank.py` adds this: `rerank(query, chunks, top_k=3)`, using `cross-encoder/ms-marco-MiniLM-L-6-v2` (`sentence-transformers`, already a dependency — same library as the embedding model, no new package).

In [2]:
from src.db import get_conn, hybrid_search
from src.embeddings import embed_texts
from src.rerank import rerank

def retrieve(query: str, top_k: int = 5) -> list[dict]:
    embedding = embed_texts([query])[0]
    with get_conn() as conn:
        return hybrid_search(conn, query, embedding, top_k=top_k)

### Case 1 — a straightforward SEC-filing claim

`hybrid_search` top-5, before reranking:

In [3]:
claim = "Apple's revenue for fiscal year ending 2025-09-27 was $416,161,000,000."

top5 = retrieve(claim, top_k=5)
for r in top5:
    print(f"{r['score']:.4f}  [{r['source']}] {r['title']}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

0.0328  [secedgar] Apple Inc. (AAPL) — Revenue
0.0315  [secedgar] Apple Inc. (AAPL) — Revenue
0.0315  [secedgar] Apple Inc. (AAPL) — Revenue
0.0300  [secedgar] Apple Inc. (AAPL) — Total assets
0.0292  [secedgar] Apple Inc. (AAPL) — Revenue


Reranked down to top-3:

In [4]:
top3 = rerank(claim, top5, top_k=3)
for r in top3:
    print(f"{r['rerank_score']:.4f}  [{r['source']}] {r['title']}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

10.2322  [secedgar] Apple Inc. (AAPL) — Revenue
8.2994  [secedgar] Apple Inc. (AAPL) — Revenue
8.2237  [secedgar] Apple Inc. (AAPL) — Revenue


### Case 2 — a fuzzier, definitional claim

Case 1 is easy — the entity/metric keywords in the claim already overlap with the right chunk, so RRF alone likely gets it right. A Wikipedia-sourced definitional claim is a better test: no numeric anchor, more chunks compete on topic similarity alone.

In [5]:
claim2 = (
    "The price-to-earnings ratio is calculated by dividing a company's "
    "market capitalization by its total revenue."
)

top5_2 = retrieve(claim2, top_k=5)
for r in top5_2:
    print(f"{r['score']:.4f}  [{r['source']}] {r['title']}")

0.0328  [wikipedia] Price–earnings ratio
0.0315  [wikipedia] Price–earnings ratio
0.0304  [wikipedia] Price–earnings ratio
0.0289  [wikipedia] Price–earnings ratio
0.0287  [wikipedia] Price–earnings ratio


In [6]:
top3_2 = rerank(claim2, top5_2, top_k=3)
for r in top3_2:
    print(f"{r['rerank_score']:.4f}  [{r['source']}] {r['title']}")

5.3848  [wikipedia] Price–earnings ratio
4.6426  [wikipedia] Price–earnings ratio
1.5628  [wikipedia] Price–earnings ratio


## What's next

- Wire `rerank()` into `verify_claim()` (`src/verifier.py`) — retrieve top-5
  via `hybrid_search`, rerank to top-3, judge on the reranked set instead of
  the raw RRF top-k
- Measure it, not just eyeball it: extend `eval/compare_retrieval.py` with a
  `hybrid_rerank` row (hit_rate@3/MRR@3) alongside the existing
  minsearch/pg_text/pg_vector/hybrid_rrf comparison
- RAGAS + LLM-as-judge evaluation (baseline vs hybrid vs hybrid+rerank) —
  the other open Phase 3 item
- Query rewriting (rewrite the claim before retrieval) — third open item

[← Back to README](../README.md)